Modelo de lenguaje ajustado (fine-tuning) sobre el texto de la publicación
Hasta ahora el texto entraba por TF-IDF (cuenta palabras) y por embeddings congelados (no saben nada de "nuevo vs usado").
Acá **re-entrenamos** un transformer (DistilBERT multilingüe) para la tarea: aprende qué partes del texto indican usado.

El input es la publicación entera "escrita": título + garantía + tipo de publicación + modo de compra + precio + stock + vendidos + medios de pago + envío. Así el modelo lee la publicación como un humano.

- Mismos 5 folds que todo lo anterior → predicciones **out-of-fold** comparables (`preds/oof_bert.npy`).
- Test: promedio de los 5 modelos de los folds (`preds/test_bert.npy`).
- Después, la notebook 08 (última celda) o la 06 lo suman al ensamble.

In [1]:
%pip install -q transformers accelerate


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np, pandas as pd, torch, time, os
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from tp3_utils import cargar_raw, SEED

MODELO = 'distilbert-base-multilingual-cased'
MAX_LEN, BATCH, EPOCHS, LR = 96, 64, 2, 4e-5
N_FOLDS_CORRER, LIMITE = 5, None
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); print('device:', device)

/Users/valentinzuppa/Desktop/Descriptiva/TP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


In [3]:
tr, te = cargar_raw()
y = (tr.condition == 'used').astype(int).values

def a_texto(df):
    pagos = df.non_mercado_pago_payment_methods.map(lambda l: ', '.join(m['description'] for m in l))
    g = df.warranty.fillna('sin dato')
    return (df.title.fillna('') + ' | garantía: ' + g.str[:60] + ' | tipo: ' + df.listing_type_id + ' | ' + df.buying_mode +
            ' | precio: ' + df.price.round(0).astype(int).astype(str) + ' ' + df.currency_id +
            ' | stock: ' + df.initial_quantity.astype(str) + ' | vendidos: ' + df.sold_quantity.astype(str) +
            ' | fotos: ' + df.pictures.map(len).astype(str) + ' | variaciones: ' + df.variations.map(len).astype(str) +
            ' | envío: ' + df['shipping.mode'] + np.where(df['shipping.free_shipping'], ' gratis', '') +
            ' | relist: ' + df.automatic_relist.astype(str) + ' | pagos: ' + pagos).tolist()

T, Tt = a_texto(tr), a_texto(te)
print(T[0]); print(T[2])

Pizarron 20x30 + Imagen + Bolsita Artesanal- Yk42artesanias | garantía: Calidad y mejor precio | tipo: bronze | buy_it_now | precio: 15 ARS | stock: 1 | vendidos: 0 | fotos: 5 | variaciones: 0 | envío: not_specified | relist: False | pagos: Transferencia bancaria, Acordar con el comprador, Efectivo
Remera Manga Larga Talle S Marca Stone Caladita Hermosa | garantía: sin dato | tipo: free | buy_it_now | precio: 100 ARS | stock: 1 | vendidos: 0 | fotos: 6 | variaciones: 1 | envío: not_specified | relist: False | pagos: Transferencia bancaria, Efectivo


In [4]:
tok = AutoTokenizer.from_pretrained(MODELO)
class DS(Dataset):
    def __init__(self, textos, labels=None): self.t, self.l = textos, labels
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return i
def collate(textos, labels):
    def f(idx):
        enc = tok([textos[i] for i in idx], truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        if labels is not None: enc['labels'] = torch.tensor([labels[i] for i in idx])
        return enc
    return f
def predecir(model, textos):
    model.eval(); out = []
    dl = DataLoader(DS(textos), batch_size=256, collate_fn=collate(textos, None))
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            out.append(torch.softmax(model(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

In [5]:
os.makedirs('preds', exist_ok=True)
oof = np.full(len(y), np.nan); test_preds = []
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(T, y))
for f, (a, b) in enumerate(folds[:N_FOLDS_CORRER]):
    t0 = time.time()
    if LIMITE: a, b = a[:LIMITE], b[:LIMITE // 3]
    Ta, ya = [T[i] for i in a], y[a].tolist()
    model = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2).to(device)
    dl = DataLoader(DS(Ta, ya), batch_size=BATCH, shuffle=True, collate_fn=collate(Ta, ya))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * len(dl) * EPOCHS), len(dl) * EPOCHS)
    for ep in range(EPOCHS):
        model.train()
        for i, batch in enumerate(dl):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad()
            if i % 200 == 0: print(f'fold {f} ep {ep} paso {i}/{len(dl)} loss {loss.item():.3f}', flush=True)
    oof[b] = predecir(model, [T[i] for i in b])
    test_preds.append(predecir(model, Tt))
    print(f'>>> fold {f}: accuracy {((oof[b] >= .5) == y[b]).mean():.4f}  ({(time.time()-t0)/60:.1f} min)', flush=True)
    del model; torch.mps.empty_cache() if device == 'mps' else None

W1018 20:07:58.522000 10901 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 18913.71it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream tas

fold 0 ep 0 paso 0/875 loss 0.688
fold 0 ep 0 paso 200/875 loss 0.312
fold 0 ep 0 paso 400/875 loss 0.341
fold 0 ep 0 paso 600/875 loss 0.211
fold 0 ep 0 paso 800/875 loss 0.217
fold 0 ep 1 paso 0/875 loss 0.177
fold 0 ep 1 paso 200/875 loss 0.155
fold 0 ep 1 paso 400/875 loss 0.219
fold 0 ep 1 paso 600/875 loss 0.340
fold 0 ep 1 paso 800/875 loss 0.237
>>> fold 0: accuracy 0.9034  (26.8 min)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7815.57it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 1 ep 0 paso 0/875 loss 0.692
fold 1 ep 0 paso 200/875 loss 0.311
fold 1 ep 0 paso 400/875 loss 0.254
fold 1 ep 0 paso 600/875 loss 0.373
fold 1 ep 0 paso 800/875 loss 0.355
fold 1 ep 1 paso 0/875 loss 0.365
fold 1 ep 1 paso 200/875 loss 0.070
fold 1 ep 1 paso 400/875 loss 0.176
fold 1 ep 1 paso 600/875 loss 0.258
fold 1 ep 1 paso 800/875 loss 0.310
>>> fold 1: accuracy 0.9079  (26.4 min)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7917.67it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 2 ep 0 paso 0/875 loss 0.638
fold 2 ep 0 paso 200/875 loss 0.393
fold 2 ep 0 paso 400/875 loss 0.274
fold 2 ep 0 paso 600/875 loss 0.313
fold 2 ep 0 paso 800/875 loss 0.332
fold 2 ep 1 paso 0/875 loss 0.349
fold 2 ep 1 paso 200/875 loss 0.268
fold 2 ep 1 paso 400/875 loss 0.110
fold 2 ep 1 paso 600/875 loss 0.396
fold 2 ep 1 paso 800/875 loss 0.161
>>> fold 2: accuracy 0.8999  (26.5 min)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7808.15it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 3 ep 0 paso 0/875 loss 0.699
fold 3 ep 0 paso 200/875 loss 0.291
fold 3 ep 0 paso 400/875 loss 0.260
fold 3 ep 0 paso 600/875 loss 0.398
fold 3 ep 0 paso 800/875 loss 0.219
fold 3 ep 1 paso 0/875 loss 0.129
fold 3 ep 1 paso 200/875 loss 0.316
fold 3 ep 1 paso 400/875 loss 0.261
fold 3 ep 1 paso 600/875 loss 0.196
fold 3 ep 1 paso 800/875 loss 0.270
>>> fold 3: accuracy 0.9030  (26.6 min)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9487.66it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 4 ep 0 paso 0/875 loss 0.691
fold 4 ep 0 paso 200/875 loss 0.224
fold 4 ep 0 paso 400/875 loss 0.247
fold 4 ep 0 paso 600/875 loss 0.321
fold 4 ep 0 paso 800/875 loss 0.266
fold 4 ep 1 paso 0/875 loss 0.204
fold 4 ep 1 paso 200/875 loss 0.304
fold 4 ep 1 paso 400/875 loss 0.193
fold 4 ep 1 paso 600/875 loss 0.177
fold 4 ep 1 paso 800/875 loss 0.113
>>> fold 4: accuracy 0.9064  (26.6 min)


In [6]:
ok = ~np.isnan(oof)
print(f'BERT accuracy OOF: {((oof[ok] >= .5) == y[ok]).mean():.4f} sobre {ok.sum()} filas')
if ok.all():
    np.save('preds/oof_bert.npy', oof); np.save('preds/test_bert.npy', np.mean(test_preds, 0))
    print('guardado preds/oof_bert.npy y preds/test_bert.npy')

BERT accuracy OOF: 0.9041 sobre 70000 filas
guardado preds/oof_bert.npy y preds/test_bert.npy
